In [2]:
from src.mlp import MLPHead
import numpy as np
from src.vocab import Vocab
from src.embedding import Embedding

In [3]:
mlp = MLPHead(embed_dim=64,output_dim=12)
dummy = np.random.randn(5,64)
out = mlp(dummy)

print(out.shape)

(5, 12)


In [4]:
import pandas as pd

train_data = pd.read_csv("data/processed/train.csv")  


In [5]:
vocabulary = Vocab(2, 400, train_data["Consumer complaint narrative"])
emb = Embedding(vocab_size=len(vocabulary.word2idx), embed_dim=64)
mlp = MLPHead(embed_dim=64, output_dim=12)

sample = train_data["Consumer complaint narrative"].iloc[:5]
batch = np.array([vocabulary.encode(text) for text in sample])
pooled = emb(batch)
logits = mlp(pooled)
print(logits.shape)   # expect (5, 12)

(5, 12)


In [6]:
from src.linear import Linear

lin_with_bias = Linear(in_dim=64, out_dim=32, bias=True)
lin_no_bias = Linear(in_dim=64, out_dim=32, bias=False)
out1 = lin_with_bias(pooled)
out2 = lin_no_bias(pooled)
print(out1.shape, out2.shape)   # both should be (5, 32)

(5, 32) (5, 32)


In [9]:
from src.batchnorm import BatchNorm1d

In [21]:
BatchN = BatchNorm1d(3)
test_dummy = np.random.randn(5,3)
test_normalized = BatchN(test_dummy)
print(test_normalized.var(axis=0))

[0.9999905  0.99998696 0.99998676]


In [22]:
from src.layernorm import LayerNorm1d

In [24]:
ln = LayerNorm1d(3)
test_normalized_layer = ln(test_dummy)
print(test_normalized_layer.var(axis=1))

[0.99998966 0.99998514 0.99994038 0.99998306 0.99999489]


In [32]:
from src.mlp import MLPHead
from src.embedding import Embedding
from src.vocab import Vocab
import numpy as np
vocabulary_list = Vocab(2,400,train_data["Consumer complaint narrative"])
embedding_head = Embedding(len(vocabulary_list.word2idx),64)

mlphead = MLPHead(64,12)
sample = train_data["Consumer complaint narrative"].iloc[:5]
batch = np.array([vocabulary_list.encode(text) for text in sample])
embedded_batch = embedding_head(batch)
mlp_result = mlphead(embedded_batch)
print(mlp_result.shape)

(5, 12)


In [38]:
set(train_data["Product"].tolist())

{'Bank account or service',
 'Consumer Loan',
 'Credit card',
 'Credit reporting',
 'Debt collection',
 'Debt or credit management',
 'Money transfer / virtual currency',
 'Mortgage',
 'Payday/title/personal loan',
 'Prepaid card',
 'Student loan',
 'Vehicle loan or lease'}

In [39]:
dictio = {}
j = 0
for i in set(train_data["Product"].tolist()):
    dictio[i] = dictio.get(i,0) + j
    j+=1

In [40]:
dictio

{'Mortgage': 0,
 'Credit card': 1,
 'Consumer Loan': 2,
 'Vehicle loan or lease': 3,
 'Debt collection': 4,
 'Bank account or service': 5,
 'Debt or credit management': 6,
 'Credit reporting': 7,
 'Prepaid card': 8,
 'Student loan': 9,
 'Money transfer / virtual currency': 10,
 'Payday/title/personal loan': 11}

In [43]:
train_data[["Product","Consumer complaint narrative"]]
train_data['targets'] = train_data["Product"].map(dictio)

In [44]:
train_data[["Product","Consumer complaint narrative","targets"]]


,Product,Consumer complaint narrative,targets
0,Mortgage,My Quicken mortgage was discharged in my Chapt...,0
1,Credit reporting,"Upon reviewing my credit file, incorrect infor...",7
2,Bank account or service,I experienced multiple unauthorized transactio...,5
3,Mortgage,XXXX XX/XX/2026 To : the Federal Consumer Fina...,0
4,Debt collection,"PORTFOLIO RECOVERY ASSOCIATES , LLC Product or...",4
...,...,...,...
86129,Debt or credit management,"Hi XXXX, this is XXXX from Leap. Your account ...",6
86130,Mortgage,My home is up for a foreclosure sale the XXXX ...,0
86131,Payday/title/personal loan,To whom it may concern : I took out a car titl...,11
86132,Credit card,I acquired the XXXX XXXX. I used it to pay ren...,1


In [45]:
dictio = {}
for idx,product in enumerate(sorted(train_data["Product"].unique())):
    dictio[product] = idx
train_data['targets'] = train_data["Product"].map(dictio)    
def get_batch(batch_size):
    
    batch_sample = train_data.sample(n=batch_size)
    X_batch = batch_sample["Consumer complaint narrative"]
    y_batch = batch_sample["targets"]
    return X_batch,y_batch

In [54]:
logits = np.random.randn(32,12)
ya = np.random.random_integers(12)
max_logits = logits.max(axis=1,keepdims=True)
e_logits = np.exp(logits-max_logits)
softmaxed = e_logits / e_logits.sum(axis=1,keepdims=True)
loss = -np.log(softmaxed[np.arange(32),ya]).mean()

# this is the Stable  Version of softmax

C:\Users\chiha\AppData\Local\Temp\ipykernel_39948\1994456130.py:2: DeprecationWarning: This function is deprecated. Please call randint(1, 12 + 1) instead
  ya = np.random.random_integers(12)


In [ ]:
logits = np.random.randn(32,12)
ya = np.random.randint(0,12,size=(1,12))
Mb = logits.max(axis=1,keepdims=True)